In [20]:
!pip install faker pandas
import random, string
import pandas as pd
from faker import Faker

fake = Faker()
random.seed(42)
Faker.seed(42)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
def rand_str(n, chars=string.ascii_letters + string.digits):
    return ''.join(random.choice(chars) for _ in range(n))

def make_email():
    t = random.choice([
        "Contact: {e}", "email me at {e}", "From: {e}", "{e}",
        "Send the report to {e} by Friday", "To: {e}", "User email: {e}"
    ])
    return t.format(e=fake.email()), "email"

def make_phone():
    t = random.choice([
        "Call me on {p}", "Phone: {p}", "{p}", "WhatsApp {p}",
        "Mobile number: {p}", "Reach me at {p} after 5pm"
    ])
    return t.format(p=fake.phone_number()), "phone"

def make_url_internal():
    t = random.choice([
        "http://192.168.{a}.{b}/admin",
        "http://10.0.{a}.{b}:8080/login",
        "https://internal.{c}.local/dashboard",
        "https://staging.{c}.com/api/v1/users",
        "http://localhost:{port}/debug",
        "https://jenkins.{c}.corp/job/deploy"
    ])
    c = fake.domain_word()
    return t.format(a=random.randint(0,255), b=random.randint(1,254),
                    c=c, port=random.randint(3000,9000)), "url_internal"

def make_credential():
    t = random.choice([
        "password={pw}", "DB_PASS = '{pw}'", "API_KEY=sk-{k}",
        "AWS_ACCESS_KEY_ID=AKIA{u}", "token: ghp_{g}",
        "Authorization: Bearer {j}", "secret_key: {k}",
        "login with user admin and pass {pw}", "PRIVATE_KEY={k}"
    ])
    text = t.format(
        pw=fake.password(length=random.randint(8, 16)),
        k=rand_str(32),
        u=rand_str(16, string.ascii_uppercase + string.digits),
        g=rand_str(36),
        j=rand_str(20) + "." + rand_str(40) + "." + rand_str(20)
    )
    return text, "credential"

def make_safe():
    # normal text plus "tricky" safe lines that look a bit like sensitive data
    kind = random.choice(["sentence", "sentence", "sentence", "public_url", "version", "misc"])
    if kind == "sentence":
        return fake.sentence(nb_words=random.randint(4, 14)), "safe"
    if kind == "public_url":
        return random.choice(["https://www.google.com", "https://github.com/" + fake.user_name(),
                              "https://www.wikipedia.org", "https://www.youtube.com/watch?v=" + rand_str(11)]), "safe"
    if kind == "version":
        return f"Version {random.randint(1,9)}.{random.randint(0,9)}.{random.randint(0,9)} released", "safe"
    return random.choice([f"Order #{random.randint(10000,99999)} shipped",
                          f"Meeting at {random.randint(1,12)}:{random.choice(['00','30'])} pm",
                          f"Total: ${random.randint(5,500)}.{random.randint(0,99):02d}",
                          "File saved successfully", "Click here to continue", "Settings updated"]), "safe"

In [22]:
def make_url_internal():
    host = random.choice([
        f"192.168.{random.randint(0,255)}.{random.randint(1,254)}",
        f"10.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}",
        f"{random.choice(['internal','staging','dev','admin','jenkins','grafana','vpn'])}.{fake.domain_word()}.{random.choice(['local','corp','internal','lan'])}",
        "localhost",
    ])
    port = random.choice(["", f":{random.randint(3000,9999)}"])
    path = random.choice(["/admin", "/login", "/dashboard", "/api/v1/users", "/debug",
                          f"/{fake.word()}/{fake.word()}", f"/api/{fake.word()}?id={random.randint(1,9999)}",
                          f"/Reports/{fake.word().capitalize()}/index.html"])
    scheme = random.choice(["http://", "https://"])
    prefix = random.choice(["", "", "Open ", "URL: ", "Visit ", "Go to "])
    return f"{prefix}{scheme}{host}{port}{path}", "url_internal"

In [23]:
counts = {make_safe: 6000, make_email: 1000, make_phone: 1000,
          make_url_internal: 800, make_credential: 1200}

rows = []
for fn, n in counts.items():
    for _ in range(n):
        text, label = fn()
        rows.append({"text": text, "label": label, "source": "faker_synthetic"})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv("../data/synthetic.csv", index=False)

print(df.shape)
print(df["label"].value_counts())
df.sample(10)

(10000, 3)
label
safe            6000
credential      1200
email           1000
phone           1000
url_internal     800
Name: count, dtype: int64


,text,label,source
8155,Consumer defense song.,safe,faker_synthetic
3079,Democrat others one voice choose development h...,safe,faker_synthetic
7304,To: owenwilliam@example.net,email,faker_synthetic
238,Contact: umaxwell@example.net,email,faker_synthetic
6808,Could bit decade opportunity month.,safe,faker_synthetic
7901,https://www.youtube.com/watch?v=cdx1seP32fN,safe,faker_synthetic
8548,API_KEY=sk-367x1H50H34pBBaFLJnpQsdGmU0ZmEGn,credential,faker_synthetic
5960,Rule play century several cultural pattern eve...,safe,faker_synthetic
407,WhatsApp 001-915-550-2613x409,phone,faker_synthetic
2968,Send the report to xdiaz@example.org by Friday,email,faker_synthetic


In [24]:
import pandas as pd
s = pd.read_csv("../data/synthetic.csv")
u = s[s["label"] == "url_internal"]["text"]
print(u.str.len().describe().round(1))
print(u.sample(8, random_state=1).tolist())

count    800.0
mean      40.1
std        9.2
min       22.0
25%       33.0
50%       39.0
75%       46.0
max       73.0
Name: text, dtype: float64
['Go to http://192.168.234.111:8626/api/make?id=4601', 'Open http://10.145.52.80/Reports/Hold/index.html', 'URL: https://10.19.174.71:5250/api/perhaps?id=6586', 'https://192.168.35.234/Reports/Foreign/index.html', 'Go to http://localhost:8625/api/once?id=4413', 'http://192.168.53.131:5715/dashboard', 'Go to http://internal.leonard.corp/debug', 'https://10.45.54.175:3150/Reports/Chance/index.html']


In [25]:
df = pd.read_csv("../data/clean_dataset.csv")
print(df[(df["label"] == "safe") & df["text"].str.contains("http")]["text"].sample(5, random_state=1).tolist())

['https://github.com/murraymichelle', 'https://github.com/courtney10', 'https://www.youtube.com/watch?v=unDuvW4yrW8', 'https://github.com/anna38', 'https://www.youtube.com/watch?v=WLoJd1y0HA9']
